In [3]:
import pandas as pd
import os 
import random

from create_groups_helpers import parse_explanations

In [4]:
wdc_small_explanations = pd.read_pickle("../../data/wdc/train_small/preprocessed_wdcproducts80cc20rnd000un_train_small_with_explanation_all_fields_4_1.pkl.gz")

wdc_small_explanations.head()

,id_left,brand_left,title_left,description_left,price_left,priceCurrency_left,specTableContent_left,cluster_id_left,id_right,brand_right,...,description_right,price_right,priceCurrency_right,specTableContent_right,cluster_id_right,pair_id,label,is_hard_negative,embedding,explanation
0,14654897,NaN,HDD 35 4TB Seagate IronWolf Pro NAS ST4000NE001,NaN,154.10,NaN,NaN,1102119,36425270,NaN,...,NaN,153.99,EUR,NaN,1102119,14654897#36425270,1,False,"[0.3275445552152033, 0.01101741506399397, 0.09...",Yes.\nattribute=brand|||importance=0.2|||value...
1,31531912,NaN,Buy Quality Replica Omega Seamaster Planet Oce...,Quality AAA Replica Omega Seamaster Planet Oce...,NaN,NaN,NaN,27649829,60397145,NaN,...,"Grafikkort, AMD Radeon RX 5500 XT Overclocked ...",2322.00,NOK,NaN,1857431,31531912#60397145,0,False,"[0.055900165776579844, 0.2703106646872746, 0.1...",No. \nattribute=brand|||importance=-0.95|||val...
2,44557157,NaN,Ubiquiti UVC-G3-FLEX-3 UniFi Protect G3 FLEX C...,BackDetailsStylish Full HD (1080p) mini turret...,$‎234.95,USD,NaN,266703,90806148,NaN,...,AAA Replica Omega Seamaster Planet Ocean 600M ...,NaN,NaN,NaN,2193117,44557157#90806148,0,False,"[-0.14620856094649143, 0.2868684206862582, 0.1...",No. \nattribute=brand|||importance=0.9|||valu...
3,49605449,Brother,Brother HL-L6300DW Business Laser Printer for ...,The Brother HL-L6300DW is the ultimate monochr...,479.98,USD,NaN,408446,36985401,NaN,...,Nánari lýsing frá framleiðanda:Barcode: 010343...,16371,ISK,NaN,126198,49605449#36985401,0,False,"[-0.24113111321950417, -0.4079377498217689, 0....",No. \nattribute=brand|||importance=0.7|||valu...
4,3024917,KINGSTON,KINGSTON 64GB USB 3.0 DataTraveler SE9 G2 (Kov...,"Lightweight, stylish USB 3.0 drive. Store, tra...",369.00,czk,NaN,435008,70174967,NaN,...,Quality AAA Replica Tag Heuer Monaco Steve McQ...,NaN,NaN,NaN,556904,3024917#70174967,0,False,"[0.020603531223475077, 0.2448852751114778, 0.4...",No. \nattribute=brand|||importance=-0.90|||va...


In [5]:
# parse the explanations
wdc_small_explanations["parsed_explanations"] = parse_explanations(wdc_small_explanations["explanation"])   

Error parsing similarity '0.10 (different currencies, but similar watch price range)' in line: attribute=price|||importance=0.1|||values=119###1899|||similarity=0.10 (different currencies, but similar watch price range). Using None.
Error parsing similarity '0.85 (prices are similar when accounting for currency conversion)' in line: attribute=price|||importance=0.05|||values=48.00 CAD###44.95 USD|||similarity=0.85 (prices are similar when accounting for currency conversion). Using None.
Error parsing similarity '0.60 (different currencies, both plausible for the product)' in line: attribute=price|||importance=0.00|||values=1.295E1 EUR###149.95 ZAR|||similarity=0.60 (different currencies, both plausible for the product). Using None.
Error parsing similarity '0.85 (price ranges align considering currency differences)' in line: attribute=price|||importance=0.05|||values=124.03 EUR###949 DKK|||similarity=0.85 (price ranges align considering currency differences). Using None.
Error parsing 

In [6]:
wdc_small_explanations["parsed_explanations"][0]

[{'attribute': 'brand',
  'importance': 0.2,
  'similarity': 1.0,
  'value1': 'Seagate',
  'value2': 'Seagate'},
 {'attribute': 'model/line',
  'importance': 0.35,
  'similarity': 1.0,
  'value1': 'IronWolf Pro',
  'value2': 'IRONWOLF PRO'},
 {'attribute': 'form factor',
  'importance': 0.2,
  'similarity': 1.0,
  'value1': '3.5 in',
  'value2': '3,5'},
 {'attribute': 'storage capacity',
  'importance': 0.3,
  'similarity': 1.0,
  'value1': '4TB',
  'value2': '4TB'},
 {'attribute': 'interface',
  'importance': 0.15,
  'similarity': 0.9,
  'value1': 'missing',
  'value2': 'SATA3'},
 {'attribute': 'rotational speed',
  'importance': 0.1,
  'similarity': 0.9,
  'value1': 'missing',
  'value2': '7200RPM'},
 {'attribute': 'buffer/cache',
  'importance': 0.1,
  'similarity': 0.9,
  'value1': 'missing',
  'value2': '128 MB'},
 {'attribute': 'part/model number',
  'importance': 0.4,
  'similarity': 0.8,
  'value1': 'ST4000NE001',
  'value2': 'missing'},
 {'attribute': 'price',
  'importance': 

In [7]:
def reconstruct_explanation_for_swap(original_parsed_explanation: list[dict], swapped_attribute_name: str) -> str:
    """
    Reconstructs the explanation string, swapping value1 and value2
    only for the specified attribute.
    """
    lines = []
    for attr_dict in original_parsed_explanation:
        current_attribute = attr_dict.get('attribute', '')
        # Determine values to use
        val1 = attr_dict.get('value1')
        val2 = attr_dict.get('value2')

        # SWAP only if this is the triggering attribute
        if current_attribute == swapped_attribute_name:
            final_val1, final_val2 = val2, val1 # Perform the swap
        else:
            final_val1, final_val2 = val1, val2 # Keep original

        # Format values part: handle None and 'missing' carefully for output string
        val1_str = final_val1 if final_val1 is not None else ''
        val2_str = final_val2 if final_val2 is not None else ''

        # Format similarity part
        sim_val = attr_dict.get('similarity')
        if sim_val is None:
            sim_str = "missing"
        else:
            # Attempt formatting, default if error (shouldn't happen if parser worked)
            try:
                 sim_str = f"{sim_val:.2f}"
            except:
                 sim_str = "missing" # Fallback

        # Format importance part
        imp_val = attr_dict.get('importance')
        if imp_val is None:
             # Decide how to represent None importance: 0.0? or keep as is? Let's use 0.00
             imp_str = "0.00"
        else:
            try:
                imp_str = f"{imp_val:.2f}"
            except:
                imp_str = "0.00" # Fallback

        parts = [
            f"attribute={current_attribute}",
            f"importance={imp_str}",
            f"values={val1_str}###{val2_str}",
            f"similarity={sim_str}"
        ]
        lines.append("|||".join(parts))

    return "\n".join(lines)


def reconstruct_explanation_for_swaps(original_parsed_explanation: list[dict], swapped_attributes: set[str]) -> str:
    """
    Reconstructs the explanation string, swapping value1 and value2
    for all specified attributes in the swapped_attributes set.
    """
    lines = []
    for attr_dict in original_parsed_explanation:
        current_attribute = attr_dict.get('attribute', '')
        # Determine values to use
        val1 = attr_dict.get('value1')
        val2 = attr_dict.get('value2')

        # SWAP if this attribute is in our set of attributes to swap
        if current_attribute in swapped_attributes:
            final_val1, final_val2 = val2, val1 # Perform the swap
        else:
            final_val1, final_val2 = val1, val2 # Keep original

        # Format values part: handle None and 'missing' carefully for output string
        val1_str = final_val1 if final_val1 is not None else ''
        val2_str = final_val2 if final_val2 is not None else ''

        # Format similarity part
        sim_val = attr_dict.get('similarity')
        if sim_val is None:
            sim_str = "missing"
        else:
            try:
                sim_str = f"{sim_val:.2f}"
            except:
                sim_str = "missing"

        # Format importance part
        imp_val = attr_dict.get('importance')
        if imp_val is None:
            imp_str = "0.00"
        else:
            try:
                imp_str = f"{imp_val:.2f}"
            except:
                imp_str = "0.00"

        parts = [
            f"attribute={current_attribute}",
            f"importance={imp_str}",
            f"values={val1_str}###{val2_str}",
            f"similarity={sim_str}"
        ]
        lines.append("|||".join(parts))

    return "\n".join(lines)

In [8]:
def augment_examples(df: pd.DataFrame, cols_to_modify_left: list[str], cols_to_modify_right: list[str]) -> pd.DataFrame:
    augmented_rows = []
    print("Starting iterative augmentation...")

    for index, row in df.iterrows():
        explanation_str = row['explanation']
        if pd.isna(explanation_str):
            continue

        # Parse the original explanation once per row
        original_parsed_explanation = parse_explanations([explanation_str])[0]
        if not original_parsed_explanation:
            continue

        row_swap_count = 0 # Counter for unique IDs generated from this single row

        # Iterate through each attribute FOUND IN THE PARSED EXPLANATION
        for attr_dict in original_parsed_explanation:
            val1 = attr_dict.get('value1')
            val2 = attr_dict.get('value2')
            attribute_name = attr_dict.get('attribute')

            # Define conditions for creating a new swapped row based on THIS attribute
            # Condition 1: Left is missing, Right has value
            if val1 == 'missing' and (val2 is not None and val2 != 'missing'):
                real_val = val2
                search_left = None
                replace_left = real_val
                search_right = real_val
                replace_right = ''
                perform_swap = True
            # Condition 2: Right is missing, Left has value
            elif val2 == 'missing' and (val1 is not None and val1 != 'missing'):
                real_val = val1
                search_left = real_val
                replace_left = ''
                search_right = None
                replace_right = real_val
                perform_swap = True
            # Condition 4: strings are defined and not the same
            elif val1 is not None and val2 is not None and val1.lower() != val2.lower():
                search_left = val1
                replace_left = val2
                search_right = val2
                replace_right = val1
                perform_swap = True
            else:
                perform_swap = False # No swap needed for this attribute


            # If a swap is indicated for THIS attribute, create a new row
            if perform_swap:
                row_swap_count += 1
                if index == 0:
                    print(f"  Row index {index} (pair_id '{row['pair_id']}'): Triggering swap for attribute '{attribute_name}' (Values: '{val1}' and '{val2}')")

                new_row = row.copy() # Copy the ORIGINAL row data for modification

                # Perform replacements in LEFT columns
                for col in cols_to_modify_left:
                    if pd.notna(new_row[col]): # Only apply to non-null values
                        try:
                            if search_left is None:
                                new_row[col] = str(new_row[col]) + " " + replace_left
                            else:
                                # Convert to string just in case, then replace
                                new_row[col] = str(new_row[col]).replace(search_left, replace_left)
                        except Exception as e:
                            print(f"    Warning: Error replacing in {col} for row {index}: {e}")


                # Perform replacements in RIGHT columns
                for col in cols_to_modify_right:
                    if pd.notna(new_row[col]):
                        try:
                            if search_right is None:
                                new_row[col] = str(new_row[col]) + " " + replace_right
                            else:
                                new_row[col] = str(new_row[col]).replace(search_right, replace_right)
                        except Exception as e:
                            print(f"    Warning: Error replacing in {col} for row {index}: {e}")

                # Update pair_id to be unique for this specific swap
                new_row['pair_id'] = f"{row['pair_id']}_swapped_{attribute_name}_{row_swap_count}"

                # Reconstruct the explanation string, swapping only the current attribute
                new_explanation_str = reconstruct_explanation_for_swap(original_parsed_explanation, attribute_name)
                new_row['explanation'] = new_explanation_str

                augmented_rows.append(new_row)

    print("Augmentation complete. Total augmented rows:", len(augmented_rows))
    return pd.DataFrame(augmented_rows)

In [9]:
def augment_examples_permutations(df: pd.DataFrame, cols_to_modify_left: list[str], cols_to_modify_right: list[str], chance_of_augmentation: float = 1) -> pd.DataFrame:
    augmented_rows = []
    print("Starting iterative augmentation...")

    for index, row in df.iterrows():
        explanation_str = row['explanation']
        if pd.isna(explanation_str):
            continue

        # Parse the original explanation once per row
        original_parsed_explanation = parse_explanations([explanation_str])[0]
        if not original_parsed_explanation:
            continue

        # Get all attributes that could potentially be swapped
        swappable_attributes = []
        for attr_dict in original_parsed_explanation:
            val1 = attr_dict.get('value1')
            val2 = attr_dict.get('value2')
            attribute_name = attr_dict.get('attribute')

            # Check if this attribute is swappable
            if (val1 == 'missing' and (val2 is not None and val2 != 'missing')) or \
               (val2 == 'missing' and (val1 is not None and val1 != 'missing')) or \
               (val1 is not None and val2 is not None and val1.lower() != val2.lower()):
                swappable_attributes.append(attribute_name)

        # Generate all possible combinations of attributes (from size 1 to all attributes)
        from itertools import combinations
        all_combinations = []
        for r in range(1, len(swappable_attributes) + 1):
            all_combinations.extend(combinations(swappable_attributes, r))

        # For each combination of attributes, create a new row
        for combo in all_combinations:
            if random.random() > chance_of_augmentation:
                    continue
            combo = set(combo)  # Convert to set for easier lookup
            if index == 0:
                print(f"  Row index {index} (pair_id '{row['pair_id']}'): Creating permutation for attributes {combo}")

            new_row = row.copy()

            # For each attribute in the combination, perform the swap
            for attribute_name in combo:
                # Find the attribute in the parsed explanation
                for attr_dict in original_parsed_explanation:
                        
                    if attr_dict.get('attribute') == attribute_name:
                        val1 = attr_dict.get('value1')
                        val2 = attr_dict.get('value2')

                        # Determine search and replace values based on the attribute's values
                        if val1 == 'missing' and (val2 is not None and val2 != 'missing'):
                            real_val = val2
                            search_left = None
                            replace_left = real_val
                            search_right = real_val
                            replace_right = ''
                        elif val2 == 'missing' and (val1 is not None and val1 != 'missing'):
                            real_val = val1
                            search_left = real_val
                            replace_left = ''
                            search_right = None
                            replace_right = real_val
                        else:  # Both values are defined and different
                            search_left = val1
                            replace_left = val2
                            search_right = val2
                            replace_right = val1

                        # Perform replacements in LEFT columns
                        for col in cols_to_modify_left:
                            if pd.notna(new_row[col]):
                                try:
                                    if search_left is None:
                                        new_row[col] = str(new_row[col]) + " " + replace_left
                                    else:
                                        new_row[col] = str(new_row[col]).replace(search_left, replace_left)
                                except Exception as e:
                                    print(f"    Warning: Error replacing in {col} for row {index}: {e}")

                        # Perform replacements in RIGHT columns
                        for col in cols_to_modify_right:
                            if pd.notna(new_row[col]):
                                try:
                                    if search_right is None:
                                        new_row[col] = str(new_row[col]) + " " + replace_right
                                    else:
                                        new_row[col] = str(new_row[col]).replace(search_right, replace_right)
                                except Exception as e:
                                    print(f"    Warning: Error replacing in {col} for row {index}: {e}")

            # Update pair_id to reflect all swapped attributes
            swapped_attrs_str = '_'.join(sorted(combo))
            new_row['pair_id'] = f"{row['pair_id']}_swapped_{swapped_attrs_str}"

            # Reconstruct the explanation string with all attributes swapped
            new_explanation_str = reconstruct_explanation_for_swaps(original_parsed_explanation, combo)
            new_row['explanation'] = new_explanation_str

            augmented_rows.append(new_row)

    print("Augmentation complete. Total augmented rows:", len(augmented_rows))
    return pd.DataFrame(augmented_rows)

## Create simple augmentation 

In [10]:
# Define columns where replacement should occur (string-based columns)
cols_to_modify_left = ['brand_left', 'title_left', 'description_left', 'price_left', 'priceCurrency_left']
cols_to_modify_right = ['brand_right', 'title_right', 'description_right', 'price_right', 'priceCurrency_right']

# only augment matching examples
matching_examples = wdc_small_explanations[wdc_small_explanations["label"] == 1]

augmented_df = augment_examples(matching_examples, cols_to_modify_left, cols_to_modify_right)

Starting iterative augmentation...
  Row index 0 (pair_id '14654897#36425270'): Triggering swap for attribute 'form factor' (Values: '3.5 in' and '3,5')
  Row index 0 (pair_id '14654897#36425270'): Triggering swap for attribute 'interface' (Values: 'missing' and 'SATA3')
  Row index 0 (pair_id '14654897#36425270'): Triggering swap for attribute 'rotational speed' (Values: 'missing' and '7200RPM')
  Row index 0 (pair_id '14654897#36425270'): Triggering swap for attribute 'buffer/cache' (Values: 'missing' and '128 MB')
  Row index 0 (pair_id '14654897#36425270'): Triggering swap for attribute 'part/model number' (Values: 'ST4000NE001' and 'missing')
  Row index 0 (pair_id '14654897#36425270'): Triggering swap for attribute 'price' (Values: '154.10' and '153.99')
  Row index 0 (pair_id '14654897#36425270'): Triggering swap for attribute 'currency' (Values: 'missing' and 'EUR')
Error parsing similarity '0.10 (different currencies, but similar watch price range)' in line: attribute=price|||

In [11]:
wdc_small_explanations.iloc[0]

id_left                                                            14654897
brand_left                                                              NaN
title_left                  HDD 35 4TB Seagate IronWolf Pro NAS ST4000NE001
description_left                                                        NaN
price_left                                                           154.10
priceCurrency_left                                                      NaN
specTableContent_left                                                   NaN
cluster_id_left                                                     1102119
id_right                                                           36425270
brand_right                                                             NaN
title_right               HD 3,5 4TB 7200RPM IRONWOLF PRO 128 MB SATA3 S...
description_right                                                       NaN
price_right                                                          153.99
priceCurrenc

In [12]:
augmented_df.iloc[0]["title_left"]

'HDD 35 4TB Seagate IronWolf Pro NAS ST4000NE001'

In [14]:
# combine the augmented examples with the original examples
combined_df = pd.concat([wdc_small_explanations, augmented_df])
# shuffle the dataframe
combined_df = combined_df.sample(frac=1).reset_index(drop=True)
#print label distribution
print(combined_df["label"].value_counts())
combined_df.to_pickle("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_explanations_41_swapped_matching_examples.pkl.gz")

label
1    3455
0    2000
Name: count, dtype: int64


## Create augementation with all permutations

In [17]:
# Define columns where replacement should occur (string-based columns)
cols_to_modify_left = ['brand_left', 'title_left', 'description_left', 'price_left', 'priceCurrency_left']
cols_to_modify_right = ['brand_right', 'title_right', 'description_right', 'price_right', 'priceCurrency_right']

# only augment matching examples
matching_examples = wdc_small_explanations[wdc_small_explanations["label"] == 1]

augmented_df = augment_examples_permutations(matching_examples, cols_to_modify_left, cols_to_modify_right, chance_of_augmentation=0.1)
combined_df = pd.concat([wdc_small_explanations, augmented_df])

# shuffle the dataframe
combined_df = combined_df.sample(frac=1).reset_index(drop=True)
#print label distribution
print(combined_df["label"].value_counts())
combined_df.to_pickle("/ceph/aasteine/fine-tuning-paper/data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_explanations_41_swapped_matching_examples_0_10_permutations.pkl.gz")


Starting iterative augmentation...
  Row index 0 (pair_id '14654897#36425270'): Creating permutation for attributes {'rotational speed'}
  Row index 0 (pair_id '14654897#36425270'): Creating permutation for attributes {'buffer/cache'}
  Row index 0 (pair_id '14654897#36425270'): Creating permutation for attributes {'form factor', 'rotational speed'}
  Row index 0 (pair_id '14654897#36425270'): Creating permutation for attributes {'price', 'interface'}
  Row index 0 (pair_id '14654897#36425270'): Creating permutation for attributes {'price', 'buffer/cache'}
  Row index 0 (pair_id '14654897#36425270'): Creating permutation for attributes {'buffer/cache', 'currency'}
  Row index 0 (pair_id '14654897#36425270'): Creating permutation for attributes {'form factor', 'part/model number', 'rotational speed'}
  Row index 0 (pair_id '14654897#36425270'): Creating permutation for attributes {'price', 'interface', 'currency'}
  Row index 0 (pair_id '14654897#36425270'): Creating permutation for att